# Notebook 01: PyTorch Training (Raw Training Loops)

3-class Sentiment Analysis for Gulf Arabic–English code-switched text.  
**Models**: XLM-RoBERTa-base, MARBERTv2 | **Strategies**: Full Fine-Tuning, LoRA (from scratch) | **Framework**: Raw PyTorch

## Section 1 — Setup

All imports, seed=42, device detection, CONFIG, and **Model Selection Justification** (per project requirements).

### MODEL SELECTION JUSTIFICATION (Section 4 Methodology)

The project spec suggests BLOOM, mT5/mBART, LaMini-FLAN-T5, AceGPT/Jais. We use **XLM-RoBERTa-base** and **MARBERTv2** instead.

- **XLM-RoBERTa**: Strongest cross-lingual encoder baseline, trained on 100 languages including Arabic and English, directly supports our code-switching task.
- **MARBERTv2**: Purpose-built for Arabic dialects, trained on 1B Arabic tweets covering Gulf dialect specifically; outperforms general multilingual models on Arabic NLP tasks.
- **AceGPT/Jais** are decoder-only generative models requiring a different architecture for classification; MARBERTv2 is the superior choice for our encoder-based sentiment task.

This is documented in Section 4 (Methodology) of the report.

In [ ]:
import os
import random
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
import yaml

# Repo root (parent of notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

config_path = ROOT / "configs" / "config.yaml"
CONFIG = yaml.safe_load(config_path.read_text()) if config_path.exists() else {}
print("Config loaded.")

## Section 2 — Data Loading and Exploration

Load master_dataset.csv, split sizes, label distributions, bar chart, 3 examples per class, text length histogram.

In [ ]:
# Dataset path: try data/processed then preprocessing/data/processed
master_path = ROOT / "data" / "processed" / "master_dataset.csv"
if not master_path.exists():
    master_path = ROOT / "preprocessing" / "data" / "processed" / "master_dataset.csv"
if not master_path.exists():
    master_path = ROOT / "preprocessing" / "data" / "processed" / "training_dataset.csv"
df = pd.read_csv(master_path)
print("Shape:", df.shape)
print("\nSplit sizes:")
print(df["split"].value_counts() if "split" in df.columns else "No split column")
print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
ax = df["label"].value_counts().sort_index().plot(kind="bar", title="Label distribution")
ax.set_xticklabels(["negative", "neutral", "positive"])
plt.tight_layout()
plt.show()

In [ ]:
# 3 example rows per class
for label in [0, 1, 2]:
    subset = df[df["label"] == label].head(3)
    print(f"--- Label {label} ---")
    for _, row in subset.iterrows():
        print(row["text"][:120] if "text" in row else row.iloc[0])
    print()

In [ ]:
# Text length histogram
lengths = df["text"].astype(str).str.len()
lengths.hist(bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Character length")
plt.title("Text length distribution")
plt.show()

## Section 3 — Back-Translation Augmentation

Import BackTranslationAugmenter, run on neutral class (50-sample demo). Why back-translation, what it adds, limitations. Full augmentation can be applied in build_dataset when config.augmentation.back_translation_enabled = true.

In [ ]:
from preprocessing.back_translation import BackTranslationAugmenter
augmenter = BackTranslationAugmenter(quality_threshold=0.6)
train_df = df[df["split"] == "train"] if "split" in df.columns else df.head(5000)
neutral_df = train_df[train_df["label"] == 1].head(50)
augmented = augmenter.augment_dataframe(neutral_df, text_col="text", label_col="label", target_classes=[1], max_samples=10)
print("Augmented rows:", len(augmented))
if len(augmented) > 0:
    print("Before:", neutral_df["text"].iloc[0][:80])
    print("After:", augmented["text"].iloc[0][:80])

## Section 4 — Tokenizer Analysis and OOV Handling

Load XLM-RoBERTa and MARBERT tokenizers; OOV rate on 500 samples; tokenization of 3 CS examples; sequence length distribution. How each tokenizer handles Arabic–English mixing and why subword tokenization addresses OOV.

In [ ]:
from transformers import AutoTokenizer
xlm_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
mar_tok = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")
samples = df["text"].astype(str).sample(500, random_state=42).tolist()
def oov_rate(tokenizer, texts):
    unk = tokenizer.unk_token_id
    ids = tokenizer(texts, truncation=True, max_length=128)["input_ids"]
    total = sum(len(x) for x in ids)
    unk_count = sum(1 for seq in ids for t in seq if t == unk)
    return unk_count / total if total else 0
print("OOV rate XLM-RoBERTa:", oov_rate(xlm_tok, samples))
print("OOV rate MARBERTv2:", oov_rate(mar_tok, samples))
cs_examples = df[df["text_type"] == "code_switched"]["text"].head(3).tolist() if "text_type" in df.columns else df["text"].head(3).tolist()
for i, ex in enumerate(cs_examples):
    print(f"Example {i+1}:", ex[:80])
    print("  XLM-RoBERTa:", xlm_tok.tokenize(ex[:100]))
    print("  MARBERT:", mar_tok.tokenize(ex[:100]))

## Section 5 — Experiment A: XLM-RoBERTa Full Fine-Tuning

Load SentimentClassifier, parameter counts, train via PyTorchSentimentTrainer, plot curves, report test metrics (Accuracy, Macro-F1, BLEU, per-class F1, confusion matrix).

In [ ]:
from models.sentiment_classifier import SentimentClassifier
from training.dataset_loader import load_splits
from training.trainer_pytorch import PyTorchSentimentTrainer
from evaluation.standard_metrics import SentimentEvaluator

max_len = CONFIG.get("data", {}).get("max_sequence_length", 128)
batch_size = CONFIG.get("training", {}).get("batch_size", 16)
train_loader, val_loader, test_loader = load_splits(master_path, xlm_tok, max_length=max_len, batch_size=batch_size)
model_a = SentimentClassifier("xlm-roberta-base", num_labels=3, dropout=0.3, freeze_base=False)
print("Parameter counts:", model_a.count_parameters())
out_dir = ROOT / "outputs" / "exp_a_xlm_full"
trainer_a = PyTorchSentimentTrainer(model_a, train_loader, val_loader, test_loader, CONFIG, "exp_a", out_dir, device=device, model_name="XLM-RoBERTa", strategy="full_finetuning")
trainer_a.train()
results_a = trainer_a.run_final_evaluation()
eval_a = SentimentEvaluator()
eval_a.plot_training_curves({"history": trainer_a.history}, out_dir / "curves.png")
print("Test Accuracy:", results_a["test_accuracy"], "Macro-F1:", results_a["test_macro_f1"], "BLEU:", results_a["test_bleu"])
print("F1 neg/neu/pos:", results_a["test_f1_negative"], results_a["test_f1_neutral"], results_a["test_f1_positive"])

## Section 6 — Experiment B: XLM-RoBERTa LoRA

Apply apply_lora_to_transformer(), verify_lora_correctness(), parameter reduction vs A, train and evaluate, same metrics.

In [ ]:
from models.lora.lora_model import apply_lora_to_transformer, verify_lora_correctness
from models.lora.lora_utils import compare_parameter_counts

model_b = SentimentClassifier("xlm-roberta-base", num_labels=3, dropout=0.3, freeze_base=False)
full_stats = model_b.count_parameters()
model_b, lora_stats = apply_lora_to_transformer(model_b.encoder, rank=8, lora_alpha=16.0, lora_dropout=0.1)
verify_lora_correctness(model_b, xlm_tok, device=str(device))
print(compare_parameter_counts(full_stats, model_b.count_parameters()))
train_loader2, val_loader2, test_loader2 = load_splits(master_path, xlm_tok, max_length=max_len, batch_size=batch_size)
out_dir_b = ROOT / "outputs" / "exp_b_xlm_lora"
trainer_b = PyTorchSentimentTrainer(model_b, train_loader2, val_loader2, test_loader2, CONFIG, "exp_b", out_dir_b, device=device, model_name="XLM-RoBERTa", strategy="lora")
trainer_b.train()
results_b = trainer_b.run_final_evaluation()
print("Test Macro-F1:", results_b["test_macro_f1"], "BLEU:", results_b["test_bleu"])

## Section 7 — Experiment C: MARBERT Full Fine-Tuning

Same as Experiment A with MARBERT.

In [ ]:
model_c = SentimentClassifier("UBC-NLP/MARBERTv2", num_labels=3, dropout=0.3, freeze_base=False)
train_loader3, val_loader3, test_loader3 = load_splits(master_path, mar_tok, max_length=max_len, batch_size=batch_size)
out_dir_c = ROOT / "outputs" / "exp_c_marbert_full"
trainer_c = PyTorchSentimentTrainer(model_c, train_loader3, val_loader3, test_loader3, CONFIG, "exp_c", out_dir_c, device=device, model_name="MARBERTv2", strategy="full_finetuning")
trainer_c.train()
results_c = trainer_c.run_final_evaluation()
print("MARBERT Full - Macro-F1:", results_c["test_macro_f1"], "BLEU:", results_c["test_bleu"])

## Section 8 — Experiment D: MARBERT LoRA

Same as Experiment B with MARBERT.

In [ ]:
model_d = SentimentClassifier("UBC-NLP/MARBERTv2", num_labels=3, dropout=0.3, freeze_base=False)
model_d, _ = apply_lora_to_transformer(model_d.encoder, rank=8, lora_alpha=16.0, lora_dropout=0.1)
train_loader4, val_loader4, test_loader4 = load_splits(master_path, mar_tok, max_length=max_len, batch_size=batch_size)
out_dir_d = ROOT / "outputs" / "exp_d_marbert_lora"
trainer_d = PyTorchSentimentTrainer(model_d, train_loader4, val_loader4, test_loader4, CONFIG, "exp_d", out_dir_d, device=device, model_name="MARBERTv2", strategy="lora")
trainer_d.train()
results_d = trainer_d.run_final_evaluation()
print("MARBERT LoRA - Macro-F1:", results_d["test_macro_f1"])

## Section 9 — Cross-Experiment Comparison

build_comparison_table with all 4 results; grouped bar chart (Accuracy, Macro-F1, BLEU); analysis: which model won, full vs LoRA, parameter count, training time.

In [ ]:
all_results = [results_a, results_b, results_c, results_d]
table = eval_a.build_comparison_table(all_results)
print(table)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(all_results))
width = 0.25
ax.bar(x - width, [r["test_accuracy"] for r in all_results], width, label="Accuracy")
ax.bar(x, [r["test_macro_f1"] for r in all_results], width, label="Macro-F1")
ax.bar(x + width, [r["test_bleu"] for r in all_results], width, label="BLEU")
ax.set_xticks(x)
ax.set_xticklabels(["A-XLM-Full", "B-XLM-LoRA", "C-Marbert-Full", "D-Marbert-LoRA"])
ax.legend()
plt.tight_layout()
plt.show()

## Section 10 — CSS Cultural Metric

Load best model, CSSEvaluator.compute_css on code_switched_only.csv, CSS score + interpretation, violin plot, top 10 highest/lowest delta examples.

In [ ]:
from evaluation.css_metric import CSSEvaluator
best_model = model_c
best_tok = mar_tok
css_eval = CSSEvaluator(best_model, best_tok, device)
cs_path = ROOT / "data" / "processed" / "code_switched_only.csv"
if not cs_path.exists():
    cs_path = ROOT / "preprocessing" / "data" / "processed" / "code_switched.csv"
cs_df = pd.read_csv(cs_path) if cs_path.exists() else df[df["text_type"] == "code_switched"].head(200)
texts_cs = cs_df["text"].astype(str).tolist()[:150]
labels_cs = cs_df["label"].astype(int).tolist()[:150]
css_out = css_eval.compute_css(texts_cs, labels_cs, n_samples=150, confidence_threshold=0.80)
print("CSS score:", css_out["css_score"], "—", css_out["interpretation"])
css_eval.plot_css_distribution(css_out["per_sample_df"], ROOT / "outputs" / "css_dist.png")
if not css_out["per_sample_df"].empty:
    top = css_out["per_sample_df"].nlargest(10, "confidence_drop")
    print("Top 10 highest delta (most Arabic reliance):"); print(top[["text_preview", "confidence_drop"]])
    low = css_out["per_sample_df"].nsmallest(10, "confidence_drop")
    print("Top 10 lowest delta:"); print(low[["text_preview", "confidence_drop"]])

## Section 11 — Error Analysis

50 misclassified examples; pattern analysis (CS text, dialect confidence, classes); failure mode discussion for report Section 7.

In [ ]:
# Error analysis: get test set predictions and misclassified examples
best_model.eval()
test_df = df[df["split"] == "test"] if "split" in df.columns else df.tail(500)
test_texts = test_df["text"].astype(str).tolist()[:500]
labels_test = test_df["label"].astype(int).tolist()[:500]
enc = mar_tok(test_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
attn = enc.get("attention_mask")
with torch.no_grad():
    out = best_model(input_ids=enc["input_ids"].to(device), attention_mask=attn.to(device) if attn is not None else None)
preds = out["logits"].argmax(dim=-1).cpu().tolist()
errs = [(test_texts[i], labels_test[i], preds[i]) for i in range(len(preds)) if labels_test[i] != preds[i]]
print("Misclassified count:", len(errs))
for i, (t, y, p) in enumerate(errs[:50]):
    print(f"{i+1}. True={y} Pred={p}: {t[:80]}...")